# Feature Importance em Árvores

Prática comparando os métodos de feature importance em modelos de árvore — MDI (`feature_importances_`), permutation importance (treino vs. teste), os três rankings nativos do XGBoost (`weight`, `gain`, `cover`) e SHAP (TreeSHAP) — sobre um dataset de regressão simulado com armadilhas plantadas de propósito: uma feature de ruído puro, uma categórica de alta cardinalidade aleatória e uma feature duplicada de uma das fortes.

## 1. Importação das bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

DRACULA_BACKGROUND = "#282a36"
DRACULA_FOREGROUND = "#f8f8f2"
DRACULA_GRID = "#44475a"

sns.set_theme(
    style="darkgrid",
    rc={
        "figure.figsize": (8, 5),
        "axes.spines.right": False,
        "axes.spines.top": False,
        "figure.facecolor": DRACULA_BACKGROUND,
        "axes.facecolor": DRACULA_BACKGROUND,
        "savefig.facecolor": DRACULA_BACKGROUND,
        "text.color": DRACULA_FOREGROUND,
        "axes.labelcolor": DRACULA_FOREGROUND,
        "xtick.color": DRACULA_FOREGROUND,
        "ytick.color": DRACULA_FOREGROUND,
        "axes.edgecolor": DRACULA_GRID,
        "grid.color": DRACULA_GRID,
    },
)

RANDOM_SEED = 42

/home/tulio_kaaz/projetos/feature-importance-arvores/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Simulação dos dados

Dataset de regressão simulando renda mensal a partir de três features legítimas (`anos_experiencia`, `anos_educacao`, `idade`), com três armadilhas plantadas de propósito:

- `ruido_puro` — ruído gaussiano sem nenhuma relação com o target, para ver se algum método lhe atribui importância não-nula.
- `codigo_agencia` — categórica de alta cardinalidade (300 códigos aleatórios), sem relação com o target, para expor o viés do MDI e do ranking `weight` do XGBoost por número de splits.
- `anos_experiencia_duplicada` — cópia exata de `anos_experiencia`, para testar como a importância se dilui entre uma feature forte e sua duplicata.

In [ ]:
def simular_idade(rng, minimo, maximo, n):
    return rng.integers(minimo, maximo + 1, n)


def simular_anos_educacao(rng, anos_disponiveis, media, desvio):
    return np.clip(rng.normal(media, desvio, len(anos_disponiveis)), 0, anos_disponiveis)


def simular_anos_experiencia(rng, anos_disponiveis, anos_educacao, media, desvio):
    limite_superior = anos_disponiveis - anos_educacao
    return np.clip(rng.normal(media, desvio, len(anos_disponiveis)), 0, limite_superior)


def simular_renda_mensal(rng, anos_experiencia, anos_educacao, idade, base, coef_experiencia, coef_educacao, coef_idade_log, noise_std):
    return (
        base
        + coef_experiencia * anos_experiencia
        + coef_educacao * anos_educacao
        + coef_idade_log * np.log1p(idade)
        + rng.normal(0, noise_std, len(idade))
    )


def montar_features(anos_experiencia, anos_educacao, idade, ruido_puro, codigo_agencia):
    return pd.DataFrame({
        "anos_experiencia": anos_experiencia,
        "anos_educacao": anos_educacao,
        "idade": idade,
        "ruido_puro": ruido_puro,
        "codigo_agencia": codigo_agencia,
        "anos_experiencia_duplicada": anos_experiencia.copy(),
    })


rng = np.random.default_rng(RANDOM_SEED)

N_SAMPLES = 3000

EXPERIENCIA_MEAN = 10
EXPERIENCIA_STD = 5
EDUCACAO_MEAN = 12
EDUCACAO_STD = 3
IDADE_MIN = 22
IDADE_MAX = 65
IDADE_INICIO_ESCOLA = 5

RENDA_BASE = 2000
COEF_EXPERIENCIA = 350
COEF_EDUCACAO = 180
COEF_IDADE_LOG = 800
RENDA_NOISE_STD = 1500

N_CATEGORIAS_AGENCIA = 300

idade = simular_idade(rng, IDADE_MIN, IDADE_MAX, N_SAMPLES)
anos_disponiveis = idade - IDADE_INICIO_ESCOLA
anos_educacao = simular_anos_educacao(rng, anos_disponiveis, EDUCACAO_MEAN, EDUCACAO_STD)
anos_experiencia = simular_anos_experiencia(rng, anos_disponiveis, anos_educacao, EXPERIENCIA_MEAN, EXPERIENCIA_STD)

renda_mensal = simular_renda_mensal(
    rng, anos_experiencia, anos_educacao, idade,
    RENDA_BASE, COEF_EXPERIENCIA, COEF_EDUCACAO, COEF_IDADE_LOG, RENDA_NOISE_STD,
)

ruido_puro = rng.normal(0, 1, N_SAMPLES)
codigo_agencia = rng.integers(0, N_CATEGORIAS_AGENCIA, N_SAMPLES)

features = montar_features(anos_experiencia, anos_educacao, idade, ruido_puro, codigo_agencia)
target = pd.Series(renda_mensal, name="renda_mensal")

features.head()

## 3. Split treino/teste

Separação de `features`/`target` em treino (80%) e teste (20%) com `train_test_split` (`random_state=RANDOM_SEED`). O conjunto de teste é o que vai expor o viés de treino do MDI e da permutation importance calculada sobre o próprio treino.

In [ ]:
TEST_SIZE = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    features, target,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
)


## 4. Treino do modelo

`xgb.XGBRegressor` (hiperparâmetros default, `random_state=RANDOM_SEED`) treinado sobre todas as features, incluindo as três armadilhas. Esse é o único modelo usado em todas as comparações desta prática — todos os métodos a seguir são calculados sobre ele.

In [ ]:
model = xgb.XGBRegressor(random_state=RANDOM_SEED)
model.fit(X_train, y_train)

## 5. MDI — `feature_importances_`

TODO: extrair o ranking de importância via `feature_importances_` do modelo treinado. Esse é o MDI (calculado no treino) — atenção especial a `ruido_puro` e `codigo_agencia`.

In [ ]:
# TODO: ranking MDI (feature_importances_)

## 6. Permutation importance — treino vs. teste

TODO: calcular `permutation_importance` duas vezes — uma sobre o conjunto de treino, outra sobre o conjunto de teste — e comparar os dois rankings. O gap entre treino e teste é o viés de treino do MDI aparecendo (ou não) de outra forma.

In [ ]:
# TODO: permutation importance no treino

In [ ]:
# TODO: permutation importance no teste

## 7. Rankings nativos do XGBoost — weight, gain, cover

TODO: extrair os três tipos de importância nativos do XGBoost (`weight`, `gain`, `cover`, via `get_booster().get_score(importance_type=...)` ou `model.get_booster().get_score()`) e comparar os três rankings entre si.

In [ ]:
# TODO: rankings weight, gain, cover

## 8. SHAP (TreeSHAP)

TODO: calcular os valores SHAP com `shap.TreeExplainer` sobre o conjunto de teste e montar o ranking por média de |SHAP| por feature.

In [ ]:
# TODO: SHAP values e ranking por média de |SHAP|

## 9. Comparação consolidada dos rankings

TODO: montar uma tabela única com a posição (ou o valor normalizado) de cada feature em todos os métodos calculados — MDI, permutation (treino), permutation (teste), weight, gain, cover, |SHAP| médio — para visualizar concordâncias e divergências lado a lado.

In [ ]:
# TODO: tabela consolidada de rankings

## 10. Insights

TODO: responder, com base nos resultados acima:

- A feature de ruído (`ruido_puro`) aparece com importância não-nula em algum método? Em quais?
- O que acontece com o ranking de `anos_experiencia` vs. `anos_experiencia_duplicada`?
- Os três rankings do XGBoost (`weight`, `gain`, `cover`) concordam entre si? Onde divergem, e por quê?
- Permutation no treino vs. no teste: o viés de treino aparece?
- SHAP corrige alguma inconsistência que os métodos anteriores mostraram?